# TESSERA v1.1, frozen encoder (128-d downstream embeddings)

This notebook follows `src/models/tessera.py` and shows how the wrapper prepares Sentinel-2 and Sentinel-1 time series for the TESSERA encoder.

Key properties:
- Uses Sentinel-2 reflectance and Sentinel-1 VV/VH inputs.
- Normalizes each modality with wrapper constants.
- Appends day-of-year as an extra channel for each modality.
- Runs the released 192-dimensional v1.1 MPC encoder and retains its published 128-dimensional downstream prefix.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
import numpy as np
from dataio.get_input import Benchmark, ModalitySeries, NativeSeries

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))

def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

## What TESSERA expects as input

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `s2_input` | `(N, T, 11)` | 10 S2 bands plus day-of-year |
| `s1_input` | `(N, T, 3)` | VV, VH plus day-of-year |
| output embedding | `(N, 128)` | pooled frozen representation |

The wrapper keeps the public `encode()` API benchmark-shaped and hides the modality-specific normalization details.

In [7]:
tessera = load('tessera_mod', 'src/models/tessera.py')

print('S2 bands:', tessera.TESSERA_S2_BANDS)
print('S1 bands:', tessera.TESSERA_S1_BANDS)
print('embedding dim:', tessera.TESSERA_EMBEDDING_DIM)
print('default weights:', tessera.DEFAULT_WEIGHTS)

S2 bands: ['B4', 'B2', 'B3', 'B8', 'B8A', 'B5', 'B6', 'B7', 'B11', 'B12']
S1 bands: ['VV', 'VH']
embedding dim: 128
default weights: /Users/akshithchowdary/Developer/Projects/org/abe/robustness/data/input/models/tessera/tessera_v1_1_mpc_encoder.pt


In [8]:
rng = np.random.default_rng(7)
N, T = 4, 18


s2_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12', 'NDVI']
s1_bands = ['VV', 'VH']
climate_bands = ['temperature', 'precipitation', 'elevation', 'slope']

months = np.arange(T, dtype=np.int64) % 12
doy = np.linspace(15, 350, T).astype(np.float32)
years = np.full(T, 2021, dtype=np.int64)

s2_values = [(rng.random((T, len(s2_bands))) * 5000).astype(np.float32) for _ in range(N)]
s1_values = [rng.normal(-12, 3, size=(T, len(s1_bands))).astype(np.float32) for _ in range(N)]
climate_values = [rng.normal(0, 1, size=(T, len(climate_bands))).astype(np.float32) for _ in range(N)]

native = NativeSeries(
    s2=ModalitySeries(s2_values, [months] * N, [doy] * N, [years] * N, s2_bands),
    s1=ModalitySeries(s1_values, [months] * N, [doy] * N, [years] * N, s1_bands),
    climate=ModalitySeries(climate_values, [months] * N, [doy] * N, [years] * N, climate_bands),
)
bench = Benchmark(
    name='synthetic',
    label_kind='multiclass',
    native=native,
    labels=np.arange(N, dtype=np.int64) % 2,
    groups=np.array(['a', 'a', 'b', 'b'], dtype=object),
    latlon=np.repeat(np.array([[11.0, 79.0]], dtype=np.float32), N, axis=0),
    label_names=['zero', 'one'],
    years=np.full(N, 2021, dtype=np.int64),
)

s2_monthly, _, _, _ = bench.monthly('s2')
s1_monthly, _, _, _ = bench.monthly('s1')
climate_monthly, _, _, _ = bench.monthly('climate')
print('s2 monthly', s2_monthly.shape, bench.s2_bands)
print('s1 monthly', s1_monthly.shape, bench.s1_bands)
print('climate monthly', climate_monthly.shape, bench.climate_bands)
print('latlon', bench.latlon.shape, 'years', bench.years.shape)

s2 monthly (4, 12, 14) ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12', 'NDVI']
s1 monthly (4, 12, 2) ['VV', 'VH']
climate monthly (4, 12, 4) ['temperature', 'precipitation', 'elevation', 'slope']
latlon (4, 2) years (4,)


## Wrapper mapping

`_prepare_streams` is the v1.1 conversion used internally by `encode`. It selects the checkpoint-bound band order, keeps valid observations, buckets sequence lengths, normalizes values with MPC statistics, and appends DOY.

In [9]:
enc = tessera.TesseraModel(device='cpu')
stream_groups = enc._prepare_streams(bench)
for bucket, (indices, s2_input, s1_input) in stream_groups.items():
    print('bucket', bucket, 'sample indices', indices.tolist())
    print('s2_input', s2_input.shape, 's1_input', s1_input.shape)
    print('s2 first sample, first timestep', np.round(s2_input[0, 0], 3))
    print('s1 first sample, first timestep', np.round(s1_input[0, 0], 3))

bucket (24, 24) sample indices [0, 1, 2, 3]
s2_input (4, 24, 11) s1_input (4, 24, 3)
s2 first sample, first timestep [-0.569  0.795  0.538  0.207  0.183 -0.563  0.406 -1.565 -0.665 -0.474
 15.   ]
s1 first sample, first timestep [ 1.161  2.939 15.   ]


## Forward pass -> embedding

The real forward pass requires the TESSERA checkpoint. Leave it disabled when all you need is to inspect input preparation.

In [10]:
RUN_REAL_FORWARD = False

enc = tessera.TesseraModel(device='cpu')
weights_path = Path(enc.weights_path or os.environ.get('TESSERA_WEIGHTS') or tessera.DEFAULT_WEIGHTS)

if not RUN_REAL_FORWARD:
    print('Set RUN_REAL_FORWARD = True to run the frozen encoder.')
elif not weights_path.exists():
    print('Skipping encode: weights not found at', weights_path)
else:
    emb = enc.encode(bench)
    print('embedding', emb.shape)
    print('first row, first 8 dims', np.round(emb[0, :8], 4))

Set RUN_REAL_FORWARD = True to run the frozen encoder.
